# IT Term Extraction — Training Notebook

This notebook builds a system that scans OJT (On-the-Job Training) documents submitted by students and
identifies IT-related terms (languages, frameworks, tools, databases, cloud platforms, concepts, etc.).

**Approach — three layers, and why:**

1. **Rule-based matching (PhraseMatcher / EntityRuler)** against a predefined CSV list of terms.
   This guarantees every term you already know about (`data/it_terms.csv`) is found and counted exactly,
   with no training required. This alone can already power the deployment script.
2. **Context-pattern mining** (`scripts/context_miner.py`, dependency-parse based) for terms that are
   **not** in the CSV — it looks for phrasing like "I integrated **Stripe API**" or "worked with
   **Terraform**" and pulls out the object regardless of whether it's on your list. Needs no
   training at all.
3. **Trainable statistical NER model** (`spaCy`'s NER component), *bootstrapped* from layers 1 and 2.
   Training on both the exact-CSV labels and the generic context-pattern labels teaches the model
   the underlying *pattern*, so it generalizes to terms it has never seen by name — not just the
   ones already in your CSV.

**Requires** `context_miner.py` in `scripts/` dir in the parent folder, plus:
```bash
pip install -r requirements.txt
```

**Outputs saved by this notebook** (used later by the deployment script):
- `models/it_term_ruler/` — rule-based-only pipeline (CSV → EntityRuler). Fast, exact, zero training.
- `models/it_term_ner/` — trained statistical NER pipeline. Generalizes beyond the exact list.
- `data/it_terms.csv` — the predefined term list (copy this next to your deployment script).


In [1]:
import json
import random
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from spacy.pipeline import EntityRuler
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.scorer import Scorer
from spacy.util import filter_spans
from tqdm.auto import tqdm

PARENT_DIR = Path("..")
sys.path.insert(0, str(Path("../scripts")))

from context_miner import load_context_pipeline, build_dep_matcher, mine_candidates

random.seed(42)

DATA_DIR = Path(PARENT_DIR / "uploads" / "train")             # put raw OJT documents (.txt/.docx/.pdf) here
TERMS_CSV = Path(PARENT_DIR / "data" / "it_terms.csv")        # predefined term list
MODELS_DIR = Path(PARENT_DIR / "models")
MODELS_DIR.mkdir(exist_ok=True)

GENERIC_LABEL = "TECH_TERM"  # generic label for context-discovered terms not yet in the CSV

/home/caineirb/Documents/Practice_Shits/spacy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Load the predefined IT terms (CSV)

Expected CSV format — two columns: (sample data only)

| term            | label        |
|-----------------|--------------|
| Python          | PROG_LANG    |
| React           | FRAMEWORK    |
| MySQL           | DATABASE     |
| Docker          | TOOL         |
| REST API        | CONCEPT      |

`label` is the entity category. If you don't care about categories yet, put the same
value (e.g. `IT_TERM` or `CLERICAL`) for every row — you can always split them later.

In [2]:
terms_df = pd.read_csv(TERMS_CSV)
terms_df["term"] = terms_df["term"].str.strip()
terms_df["label"] = terms_df["label"].str.strip().str.upper()
assert {"term", "label"}.issubset(terms_df.columns), "CSV must have 'term' and 'label' columns"
print(f"Loaded {len(terms_df)} predefined terms across {terms_df['label'].nunique()} categories")
terms_df.head()

Loaded 41 predefined terms across 8 categories


,term,label
0,Python,PROG_LANG
1,Java,PROG_LANG
2,JavaScript,PROG_LANG
3,TypeScript,PROG_LANG
4,C#,PROG_LANG


## Step 2 — Build the rule-based matcher (EntityRuler)

This is the exact-match backbone. It never needs "training" — it just needs the CSV.
We build it first because we'll reuse it in Step 4 to auto-annotate training data.


In [3]:
def build_ruler_pipeline(terms_df: pd.DataFrame) -> spacy.language.Language:
    nlp = spacy.blank("en")
    ruler = nlp.add_pipe("entity_ruler", config={"phrase_matcher_attr": "LOWER"})
    patterns = [
        {"label": row.label, "pattern": row.term}
        for row in terms_df.itertuples()
    ]
    ruler.add_patterns(patterns)
    return nlp

ruler_nlp = build_ruler_pipeline(terms_df)

# quick smoke test
sample = "During my OJT I used Python and Django with a MySQL database, deployed via Docker on AWS."
doc = ruler_nlp(sample)
[(ent.text, ent.label_) for ent in doc.ents]


[('Python', 'PROG_LANG'),
 ('Django', 'FRAMEWORK'),
 ('MySQL', 'DATABASE'),
 ('Docker', 'TOOL'),
 ('AWS', 'CLOUD')]

## Step 3 — Load the raw OJT documents

Point `DATA_DIR` at a folder containing the students' submitted documents. Supports
`.txt`, `.docx`, and `.pdf`. This is the *unlabeled* corpus we'll auto-annotate next.

<div class="alert alert-block alert-warning">
    <b>Note:</b> 
Since some documents were just scanned (like using CamScanner), common pdf reader wont read the documents as it is, so you need to use some kind of  Optical Character Recognition (OCR) tool to process the pixels into readable characters. This part is tricky as the computer which you will be training this one need to have these tools:

For Windows:
1. Download and run the Tesseract installer. Note its installation path (usually `C:\Program Files\Tesseract-OCR\tesseract.exe`).
2. Download poppler-windows, extract it, and add the bin folder to your system's Environment PATH variables.

For Linux and Docker Containers:
<pre>
sudo apt update
sudo apt install tesseract-ocr poppler-utils libtesseract-dev
</pre>
</div>

In [4]:
# Directory to cache extracted text on disk (one .txt per document)
TEXT_CACHE_DIR = Path(PARENT_DIR / "_text_cache")
TEXT_CACHE_DIR.mkdir(exist_ok=True)

COLUMN_SPLIT_RATIO = 0.40  # left column = 40% of page width; tune if your forms differ

def _ocr_page_columns(page_img, split_ratio=COLUMN_SPLIT_RATIO):
    """OCR a page by splitting into left/right halves to prevent column interleaving.

    OJT log sheets have a two-column layout (task list | reflection narrative).
    Tesseract reads straight across both columns, mashing unrelated text together.
    Cropping each half independently produces clean, non-interleaved text.
    """
    import pytesseract
    w, h = page_img.size
    split_x = int(w * split_ratio)
    overlap = int(w * 0.02)  # ~2% overlap avoids cutting through boundary text

    left_crop = page_img.crop((0, 0, min(split_x + overlap, w), h))
    right_crop = page_img.crop((max(split_x - overlap, 0), 0, w, h))

    left_text = pytesseract.image_to_string(left_crop)
    right_text = pytesseract.image_to_string(right_crop)

    del left_crop, right_crop
    return left_text.strip() + "\n\n" + right_text.strip()

def ocr_scanned_pdf(pdf_path) -> str:
    from pdf2image import convert_from_path, pdfinfo_from_path
    import pytesseract

    # Get page count without loading any images into memory
    info = pdfinfo_from_path(str(pdf_path))
    total_pages = info["Pages"]
    print(f"Processing {total_pages} page(s) with OCR (one at a time, 200 DPI)...")

    extracted_text = []

    # Process ONE page at a time to keep RAM low
    for page_num in range(1, total_pages + 1):
        # convert_from_path with first_page=last_page loads only 1 page
        pages = convert_from_path(
            str(pdf_path), dpi=200,
            first_page=page_num, last_page=page_num,
        )
        text = _ocr_page_columns(pages[0])
        extracted_text.append(text)
        del pages, text  # free the PIL image + string immediately

    result = "\n".join(extracted_text)
    del extracted_text
    return result

def extract_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".txt":
        return path.read_text(encoding="utf-8", errors="ignore")
    if suffix == ".docx":
        import docx
        d = docx.Document(str(path))
        return "\n".join(p.text for p in d.paragraphs)
    if suffix == ".pdf":
        return ocr_scanned_pdf(path)
    raise ValueError(f"Unsupported file type: {suffix}")

# ── Text cleaning ────────────────────────────────────────────────────
import re
import unicodedata

# Curly quotes / dashes / non-breaking spaces that Unicode normalization alone
# won't fold to plain ASCII -- map them explicitly so meaning isn't just deleted.
_PUNCT_MAP = {
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-", "\u2026": "...", "\u00a0": " ",
}

def clean_text(text: str) -> str:
    """Normalize raw extracted/OCR'd text before it's cached or used downstream:
    - Unicode-normalizes (folds ligatures/odd glyphs into standard forms)
    - de-hyphenates words split across a line break by OCR/PDF wrapping
    - maps common "smart" punctuation to plain ASCII equivalents
    - un-escapes literal backslash-n / backslash-t sequences into real whitespace
    - drops control characters and non-English/non-ASCII noise (stray symbols,
      other scripts, OCR garbage)
    - collapses runs of whitespace and blank lines

    NOTE: this keeps ASCII only. If your documents legitimately mix in Filipino/
    Taglish or other Latin-script text you want to preserve rather than strip,
    widen the final regex below from `[^\\x00-\\x7F]+` to a broader Unicode range,
    e.g. `[^\\x00-\\x7F\\u00C0-\\u024F]+`.
    """
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)

    for bad, good in _PUNCT_MAP.items():
        text = text.replace(bad, good)

    # de-hyphenate line-wrapped words: "inte-\ngrated" -> "integrated"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # literal backslash-escaped newlines/tabs (not real whitespace) -> real whitespace
    text = text.replace("\\n", "\n").replace("\\t", " ")

    # drop control characters, keeping newline/tab (collapsed below)
    text = "".join(ch for ch in text if ch in "\n\t" or unicodedata.category(ch)[0] != "C")

    # drop non-English/non-ASCII noise (see docstring note above to widen this)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    # collapse whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # drop lines that are just 1-2 non-word characters or isolated digits (OCR debris)
    text = "\n".join(
        ln for ln in text.split("\n")
        if len(ln.strip()) > 2 or re.match(r"[A-Za-z]{2,}", ln.strip())
    )
    return text.strip()

# ── OCR pipe → I fix ─────────────────────────────────────────────────
def fix_ocr_pipe_to_I(text: str) -> str:
    """Fix common OCR misread where 'I' is recognized as '|'.

    Rules (contextual — preserves legitimate pipe usage):
    - '|' at the start of a line followed by a space + lowercase → 'I'
      (e.g. "| also updated" → "I also updated")
    - '|' mid-sentence where surrounded by spaces and next word is lowercase
      (e.g. "and | have" → "and I have"), but NOT after date patterns
    - '|' immediately before a lowercase letter → 'I'
      (e.g. "|learned" → "Ilearned")
    - Preserves '|' after date patterns like "Feb 23 |" (pipe used as column separator)
    """
    if not text:
        return text

    # Line-start: "| also" → "I also", "| am" → "I am"
    text = re.sub(r"^\| (?=[a-z])", "I ", text, flags=re.MULTILINE)

    # Mid-sentence: " | word" where word starts lowercase → " I word"
    # But NOT after date-like patterns (e.g. "Feb 23 |", "Day 1 |", "Mar 02 |")
    # Use a callback since Python doesn't support variable-width lookbehinds
    _date_before_pipe = re.compile(r"[A-Z][a-z]{2}\s+\d{1,2}\s*$")
    _day_before_pipe = re.compile(r"Day\s+\d+\s*$")
    def _pipe_replacer(m):
        before = text[:m.start()]
        if _date_before_pipe.search(before) or _day_before_pipe.search(before):
            return m.group(0)  # keep original — it's a date separator
        return " I "
    text = re.sub(r" \| (?=[a-z])", _pipe_replacer, text)

    # "|" immediately before lowercase letter: "|learned" → "Ilearned"
    text = re.sub(r"\|(?=[a-z])", "I", text)

    # "|_" → "I_"
    text = re.sub(r"\|(?=[_])", "I", text)

    return text

# ── Boilerplate stripping (section-based extraction) ─────────────────
# These patterns match institutional headers, signatures, form structure,
# and OCR-garbled variants of the same. Used as a second-pass filter after
# section-based extraction.
_BOILERPLATE_PATTERNS = [
    # Institutional headers (with OCR variation tolerance)
    re.compile(r"^.*Republic\s+(of|ol|ot|ef|o)\s+(the|te|ti|i?he)\s+(Ph|Pl|Pr|Pi|ei)", re.IGNORECASE),
    re.compile(r"^.*NORTH\w*\s+BUKIDNON", re.IGNORECASE),
    re.compile(r"^.*BUKIDNON\s+STATE\s*(CO?LL?\w*|OLLL?\w*)", re.IGNORECASE),
    re.compile(r"^.*Manolo\s*Fort?\w*[,.]?\s*\d*\s*Buki", re.IGNORECASE),
    re.compile(r"^.*TANKULAN.*MANOLO\s*FORT", re.IGNORECASE),
    re.compile(r"^.*New\s+Government\s+Center.*Manolo", re.IGNORECASE),

    # Form structure headers
    re.compile(r"^.*ON[-~\s]*THE[-~\s]*JOB\s+TRAINING", re.IGNORECASE),
    re.compile(r"^.*WEEKLY\s+PROGRESS\s+RE", re.IGNORECASE),
    re.compile(r"^\s*Form\s*\d+\s*$", re.IGNORECASE),
    re.compile(r"^\s*ACTIVITIES:\s*$", re.IGNORECASE),
    re.compile(r"^\s*REFLECTIONS?:\s*$", re.IGNORECASE),
    re.compile(r"^\s*ACTIVITIES:\s*REFLECTIONS?:\s*$", re.IGNORECASE),

    # Signature blocks
    re.compile(r"^.*Documentation:\s*\d*\s*$", re.IGNORECASE),
    re.compile(r"^.*Signed:\s*", re.IGNORECASE),
    re.compile(r"^.*S(t[uyd]*\w*|)\s*[Tt]rainee\s+[Ss]ignature", re.IGNORECASE),
    re.compile(r"^.*Signature\s+over\s+Pr", re.IGNORECASE),
    re.compile(r"^.*Name\s+of\s+(HTE|THE|MTE)\s+Supervisor", re.IGNORECASE),

    # OJT coordinator lines (with massive OCR variation)
    re.compile(r"^\s*[O0Q@]\w{0,2}T?\s*Co?\w{0,2}r?d?\w*[aio]?\w*r?\s*[\W]*$", re.IGNORECASE),
    re.compile(r"^\s*\w{0,4}\s*OJT\s*Co\w*r?\w*\s*[\W]*$", re.IGNORECASE),
    re.compile(r"^\s*Coordinator\s*$", re.IGNORECASE),

    # Standalone "Student Trainee" (without Signature — already handled above with Signature)
    re.compile(r"^\s*Student\s*Trainee\s*$", re.IGNORECASE),

    # Known personnel names that appear as signatures
    re.compile(r"^.*SHIELA.*O\w*ZCO", re.IGNORECASE),
    re.compile(r"^.*FAISAH.*BACARAT", re.IGNORECASE),
    re.compile(r"^.*OROOZCO", re.IGNORECASE),

    # Form metadata lines
    re.compile(r"^\s*Term\s+Second\s+Semester", re.IGNORECASE),
    re.compile(r"^\s*Term\s+\w+\s+Semester\s+[AS]Y", re.IGNORECASE),
    re.compile(r"^\s*Week\s+Week\s*\d", re.IGNORECASE),
    re.compile(r"^\s*Week\s*\d+\s*\(", re.IGNORECASE),
    re.compile(r"^\s*(Name|Company|Week)\s+[A-Z]", re.IGNORECASE),

    # OCR garbage lines (mostly non-alphanumeric, or very short all-caps gibberish)
    re.compile(r"^[\W\d\s]{5,}$"),  # lines that are mostly symbols/digits
    re.compile(r"^[^a-zA-Z]*$"),     # lines with no letters at all

    # OCR debris: lines where letters are mostly isolated single chars with spaces/symbols
    # e.g. "e R . --- O P e RTINS g g ---", "i vitae, Puctac a", "V1. Dt & Wi"
    re.compile(r"^(?:\s*[\W]*\s*[A-Za-z]{1,2}\s+){3,}"),  # 3+ isolated 1-2 char clusters

    # Questionnaire/evaluation boilerplate
    re.compile(r"^.*comments\s+as\s+needed.*Return\s+completed", re.IGNORECASE),
    re.compile(r"^.*Has\s+the\s+(Unit\s+)?Coordinator\s+provided", re.IGNORECASE),
    re.compile(r"^.*Has\s+the\s+supervision\s+of\s+the", re.IGNORECASE),
    re.compile(r"^.*successfully\s+completed\s+the\s+\d+\s+hours\s+of", re.IGNORECASE),

    # Standalone section labels (keep these only when they have content after)
    re.compile(r"^\s*Objective\(s\)\s+for\s+the\s+week:\s*$", re.IGNORECASE),
]

def _is_boilerplate(line: str) -> bool:
    """Check if a single line matches any known boilerplate pattern."""
    stripped = line.strip()
    if not stripped:
        return False  # blank lines handled separately
    # Very short lines that are mostly uppercase and non-narrative
    if len(stripped) < 4 and not re.match(r"[A-Za-z]{3,}", stripped):
        return True
    return any(pat.match(stripped) for pat in _BOILERPLATE_PATTERNS)

def strip_boilerplate(text: str) -> str:
    """Extract only student activity descriptions and reflections from OJT log text.

    Strategy (section-based extraction):
    1. Split the document into weekly sections by the repeating 'ON-THE-JOB
       TRAINING LOG SHEET' header.
    2. Within each section, locate the ACTIVITIES/REFLECTIONS content area.
    3. Extract narrative paragraphs, discarding form headers, signatures,
       institutional boilerplate, and OCR garbage.
    4. Apply line-level boilerplate filter as a second pass to catch any
       remaining noise.

    Returns the cleaned narrative text, or empty string if nothing survives.
    """
    if not text:
        return ""

    # Split into weekly sections using the log sheet header
    # (tolerant of OCR variations in the header)
    section_pattern = re.compile(
        r"ON[-~\s]*THE[-~\s]*JOB\s+TRAINING\s+(LOG\s+SHEET|EXECUTIVE)",
        re.IGNORECASE,
    )
    section_starts = [m.start() for m in section_pattern.finditer(text)]

    if not section_starts:
        # No sections found — apply line-level filter to the whole text
        lines = text.split("\n")
        kept = [ln for ln in lines if not _is_boilerplate(ln)]
        return "\n".join(kept).strip()

    # Build section boundaries
    sections = []
    for i, start in enumerate(section_starts):
        end = section_starts[i + 1] if i + 1 < len(section_starts) else len(text)
        sections.append(text[start:end])

    # ── Extract narrative from each section ──
    narrative_chunks = []
    # Pattern to detect the start of activities/reflections content
    content_start_re = re.compile(
        r"(?:ACTIVITIES:\s*(?:REFLECTIONS?:)?|REFLECTIONS?:)",
        re.IGNORECASE,
    )
    # Pattern to detect boilerplate footer that ends content
    footer_re = re.compile(
        r"(?:Republic\s+(?:of|ol|ot|ef|o)\s+(?:the|te|ti)|Documentation:\s*\d*$|Signed:)",
        re.IGNORECASE,
    )

    for section in sections:
        # Find where the narrative content starts
        content_match = content_start_re.search(section)
        if content_match:
            # Start from after the ACTIVITIES:/REFLECTIONS: label
            narrative_start = content_match.end()
        else:
            # No explicit marker — try starting after the objective line
            obj_match = re.search(r"Objective\(s\)\s+for\s+the\s+week:.*?\n", section, re.IGNORECASE)
            if obj_match:
                narrative_start = obj_match.end()
            else:
                # Skip this section — it's probably just a header page
                continue

        narrative_text = section[narrative_start:]

        # Find where boilerplate footer starts (end of narrative)
        footer_match = footer_re.search(narrative_text)
        if footer_match:
            narrative_text = narrative_text[:footer_match.start()]

        # Apply line-level boilerplate filter
        lines = narrative_text.split("\n")
        kept_lines = [ln for ln in lines if not _is_boilerplate(ln)]
        chunk = "\n".join(kept_lines).strip()
        if chunk:
            narrative_chunks.append(chunk)

    result = "\n\n".join(narrative_chunks)

    # Final cleanup: collapse excessive blank lines
    result = re.sub(r"\n{3,}", "\n\n", result)
    return result.strip()

LOG_SHEET_HEADER = "ON-THE-JOB TRAINING LOG SHEET"

def trim_to_log_sheet(text: str) -> str | None:
    """Keep text starting from the first 'ON-THE-JOB TRAINING LOG SHEET',
    removing all preamble text before it. Returns None if the header is not found.
    """
    idx = text.find(LOG_SHEET_HEADER)
    if idx == -1:
        # Fallback to case-insensitive regex for slight spacing/casing variations
        m = re.search(r"ON[-\s]*THE[-\s]*JOB\s+TRAINING\s+LOG\s+SHEET", text, re.IGNORECASE)
        if m:
            idx = m.start()
    if idx != -1:
        return text[idx:].strip()
    return None

# ── Discover documents and extract → clean → cache to disk ───────────
doc_paths = sorted(
    p for p in DATA_DIR.glob("**/*") if p.suffix.lower() in {".txt", ".docx", ".pdf"}
) if DATA_DIR.exists() else []
print(f"Found {len(doc_paths)} documents in {DATA_DIR}")

# ── Stream extracted text to disk instead of keeping it all in RAM ──
# Each document gets a .txt cache file; only one document's text is in
# memory at a time during extraction.
text_cache_paths: list[Path] = []
for doc_path in tqdm(doc_paths, desc="Extracting text → disk"):
    # Check for an existing cache file for this document
    candidates = sorted(TEXT_CACHE_DIR.glob(f"{doc_path.stem}_*.txt"))
    same_stem_docs = [p for p in doc_paths if p.stem == doc_path.stem]
    if len(candidates) == len(same_stem_docs):
        cache_file = candidates[same_stem_docs.index(doc_path)]
    elif candidates:
        cache_file = candidates[0]
    else:
        cache_file = TEXT_CACHE_DIR / f"{doc_path.stem}_{hash(str(doc_path)) & 0xFFFF:04x}.txt"

    if not cache_file.exists():          # skip if already cached from a previous run
        text = extract_text(doc_path)
        text = clean_text(text)          # ← normalize before caching
        text = fix_ocr_pipe_to_I(text)   # ← fix OCR | → I
        cleaned_text = trim_to_log_sheet(text)
        if cleaned_text is not None:
            cleaned_text = strip_boilerplate(cleaned_text)  # ← strip boilerplate
            if cleaned_text:
                cache_file.write_text(cleaned_text, encoding="utf-8")
                text_cache_paths.append(cache_file)
        else:
            # File doesn't contain a log sheet; delete cache file if it exists and skip
            if cache_file.exists():
                cache_file.unlink()
        del text, cleaned_text           # free RAM immediately
    else:
        # Re-clean existing cached text through the full pipeline
        cached_text = cache_file.read_text(encoding="utf-8")
        cleaned_text = fix_ocr_pipe_to_I(cached_text)
        cleaned_text = strip_boilerplate(cleaned_text)
        if cleaned_text and cleaned_text.strip():
            if cleaned_text != cached_text:
                cache_file.write_text(cleaned_text, encoding="utf-8")
            text_cache_paths.append(cache_file)
        else:
            cache_file.unlink(missing_ok=True)
        del cached_text, cleaned_text

print(f"Cached {len(text_cache_paths)} document texts in {TEXT_CACHE_DIR}/")
print("(raw_texts list is NOT held in RAM — downstream cells read from disk)")


## Step 4 — Mine context-pattern candidates for terms *not* in the CSV

This is the piece that lets the model go beyond your predefined list. Students will
mention real tools/tech you haven't added to `it_terms.csv` yet — this step finds
them by looking at **how** the word is used, not whether it's on a list.

It looks for phrasing like:

- "I **integrated** Stripe API..."
- "...**deployed** using Kubernetes..."
- "we **migrated** to PostgreSQL"
- "**worked with** Terraform"

i.e. a trigger verb (integrate, use, deploy, migrate, configure, work with, ...)
followed by a noun phrase — and returns that noun phrase as a candidate term,
filtering out generic non-tech objects ("used my skills", "handled the project").

This needs a dependency parser (a blank pipeline has none), so it uses
`en_core_web_sm`:
```
python -m spacy download en_core_web_sm
```


In [13]:
context_nlp = load_context_pipeline("en_core_web_sm")
dep_matcher = build_dep_matcher(context_nlp)
known_terms_lower = set(terms_df["term"].str.lower())

# Process one document at a time from disk to keep RAM low
mined_per_doc = []
for cache_path in tqdm(text_cache_paths, desc="Mining context-pattern candidates"):
    text = cache_path.read_text(encoding="utf-8")
    mined_per_doc.append(mine_candidates(text, context_nlp, dep_matcher, known_terms_lower))
    del text  # free after processing

all_candidates = [c for doc_cands in mined_per_doc for c in doc_cands]
candidate_counts = Counter(c.text for c in all_candidates)
print(f"Mined {len(candidate_counts)} distinct candidate terms not in the CSV, "
      f"across {len(text_cache_paths)} documents")

display(pd.DataFrame(candidate_counts.most_common(30), columns=["term", "doc_count"]))

candidates_df = pd.DataFrame(
    candidate_counts.most_common(),
    columns=["term", "count"]
)

candidates_df.to_csv(PARENT_DIR / "data" / "candidate_counts.csv", index=False)

Mining context-pattern candidates: 100%|██████████| 68/68 [00:28<00:00,  2.41it/s]

Mined 274 distinct candidate terms not in the CSV, across 68 documents


,term,doc_count
0,Excel,15
1,Canva,14
2,Microsoft Excel,8
3,Microsoft Word,3
4,Photoshop,3
5,UPS,2
6,Spatie,2
7,Figma prototype,2
8,Spatie Laravel Permission,2
9,UI prototypes,2


**Review this list.** Two things to do with it:

1. **Right now**: anything that's clearly a real tool/tech — add it to `it_terms.csv`
   with an appropriate label. It'll be picked up as an exact match immediately, no
   retraining needed.
2. **For training**: even the ones you haven't triaged yet are used below as a
   generic `TECH_TERM` label (not a specific category) so the NER model learns the
   *pattern* — "things introduced by these verbs tend to be tech terms" — rather
   than only memorizing your exact vocabulary. That's what gives it a shot at
   recognizing genuinely new terms in future documents.


## Step 5 — Auto-annotate a training set (silver-standard labels)

Combine two label sources into one training set:
- **Exact CSV matches** (`ruler_nlp`) → their specific category label (`PROG_LANG`, `FRAMEWORK`, ...)
- **Context-pattern candidates** (Step 4) not already covered by the CSV → generic `TECH_TERM` label

Where the two overlap, the CSV's specific label wins.

> **Recommended:** review a sample of the auto-generated annotations before training
> (see Step 5b) — silver labels are only as good as your CSV + patterns, and a quick
> manual pass catches false positives and fills gaps.


In [6]:
def _clean_entity_spans(text, ents):
    """Strip leading/trailing whitespace & punctuation from entity spans.
    Drops any span that becomes empty after stripping."""
    cleaned = []
    for start, end, label in ents:
        # Strip leading whitespace/punctuation
        while start < end and (text[start].isspace() or text[start] in '.,;:!?()[]{}"\'\\'):
            start += 1
        # Strip trailing whitespace/punctuation
        while end > start and (text[end - 1].isspace() or text[end - 1] in '.,;:!?()[]{}"\'\\'):
            end -= 1
        if start < end and text[start:end].strip():
            cleaned.append((start, end, label))
    return cleaned

def make_enriched_examples(nlp_blank, text_cache_paths, mined_per_doc):
    examples = []
    for cache_path, mined in zip(text_cache_paths, mined_per_doc):
        text = cache_path.read_text(encoding="utf-8")
        if not text or not text.strip():
            continue
        doc = nlp_blank.make_doc(text)
        ruler_ents = list(ruler_nlp(text).ents)

        context_spans = []
        for cand in mined:
            offset = 0
            while True:
                idx = text.find(cand.text, offset)
                if idx == -1:
                    break
                span = doc.char_span(idx, idx + len(cand.text), label=GENERIC_LABEL, alignment_mode="contract")
                if span is not None:
                    context_spans.append(span)
                offset = idx + len(cand.text)

        # CSV (specific-label) spans win over generic context spans on overlap
        combined = sorted(list(ruler_ents) + context_spans, key=lambda s: (s.start, -(s.end - s.start)))
        combined = filter_spans(combined)

        raw_ents = [(s.start_char, s.end_char, s.label_) for s in combined]
        clean_ents = _clean_entity_spans(text, raw_ents)

        example = Example.from_dict(doc, {"entities": clean_ents})
        examples.append(example)
        del text, doc
    return examples

blank_nlp = spacy.blank("en")
all_examples = make_enriched_examples(blank_nlp, text_cache_paths, mined_per_doc)
print(f"Built {len(all_examples)} enriched training examples "
      f"(CSV terms + context-pattern candidates) from {len(text_cache_paths)} documents")


Built 68 enriched training examples (CSV terms + context-pattern candidates) from 68 documents


### Step 4b — Load supplemental training sentences

Additional hand-written training sentences from `data/supplemental_sentences/*.json`.
These supplement the OCR-derived training data with clean, curated examples.

**Two formats supported (Option C):**
- **Annotated**: `{"text": "...", "entities": [[start, end, "LABEL"], ...]}` → used as-is (gold labels)
- **Plain text**: `{"text": "..."}` → auto-annotated by the EntityRuler + context miner (silver labels)


In [ ]:
# ── Step 4b — Load supplemental training sentences ────────────────────
SUPPLEMENTAL_DIR = Path(PARENT_DIR / "data" / "supplemental_sentences")

def load_supplemental_examples(nlp_blank, supp_dir, ruler_nlp_fn,
                                context_nlp_fn=None, dep_matcher_fn=None,
                                known_terms_lower_set=None):
    """Load hand-written training sentences from JSON files.

    Supports both NER entities and TextCat classification (IT_TASK vs CLERICAL).
    """
    examples = []
    if not supp_dir.exists():
        return examples

    json_files = sorted(supp_dir.glob("*.json"))
    if not json_files:
        return examples

    for json_path in json_files:
        with json_path.open("r", encoding="utf-8") as f:
            entries = json.load(f)
        if not isinstance(entries, list):
            entries = [entries]

        for entry in entries:
            text = entry.get("text", "").strip()
            if not text:
                continue

            doc = nlp_blank.make_doc(text)

            if "entities" in entry and entry["entities"]:
                raw_ents = [(s, e, l) for s, e, l in entry["entities"]]
            else:
                ruler_ents = list(ruler_nlp_fn(text).ents)
                context_spans = []
                if context_nlp_fn and dep_matcher_fn and known_terms_lower_set is not None:
                    mined = mine_candidates(text, context_nlp_fn, dep_matcher_fn, known_terms_lower_set)
                    for cand in mined:
                        offset = 0
                        while True:
                            idx = text.find(cand.text, offset)
                            if idx == -1:
                                break
                            span = doc.char_span(idx, idx + len(cand.text),
                                                  label=GENERIC_LABEL, alignment_mode="contract")
                            if span is not None:
                                context_spans.append(span)
                            offset = idx + len(cand.text)

                combined = sorted(list(ruler_ents) + context_spans,
                                   key=lambda s: (s.start, -(s.end - s.start)))
                combined = filter_spans(combined)
                raw_ents = [(s.start_char, s.end_char, s.label_) for s in combined]

            clean_ents = _clean_entity_spans(text, raw_ents)
            ex_data = {"entities": clean_ents}
            if "cats" in entry and entry["cats"]:
                ex_data["cats"] = entry["cats"]
            example = Example.from_dict(doc, ex_data)
            examples.append(example)

    return examples

supplemental_examples = load_supplemental_examples(
    blank_nlp, SUPPLEMENTAL_DIR, ruler_nlp,
    context_nlp_fn=context_nlp, dep_matcher_fn=dep_matcher,
    known_terms_lower_set=known_terms_lower,
)

if supplemental_examples:
    all_examples.extend(supplemental_examples)
    print(f"Loaded {len(supplemental_examples)} supplemental training examples "
          f"→ total now {len(all_examples)}")
else:
    print(f"No supplemental sentences found in {SUPPLEMENTAL_DIR}/ ")


### Step 5b (optional but recommended) — export for manual review


In [7]:
review_path = Path(PARENT_DIR / "data" / "annotations_for_review.jsonl")
with review_path.open("w", encoding="utf-8") as f:
    for ex in all_examples:
        text = ex.reference.text
        ents = [(e.start_char, e.end_char, e.label_, e.text) for e in ex.reference.ents]
        f.write(json.dumps({"text": text, "entities": ents}) + "\n")
print(f"Wrote {review_path} — review/correct, then reload before training if desired")

# To reload corrected annotations later, rebuild Examples from the corrected spans
# rather than re-running make_enriched_examples (which would just regenerate the same silver labels).


Wrote ../data/annotations_for_review.jsonl — review/correct, then reload before training if desired


## Step 6 — Train / dev split


In [8]:
random.shuffle(all_examples)
split = int(len(all_examples) * 0.8)
train_examples, dev_examples = all_examples[:split], all_examples[split:]
print(f"Train: {len(train_examples)}  Dev: {len(dev_examples)}")

Train: 54  Dev: 14


## Step 7 — Train the statistical NER model

Starts from a blank English pipeline, adds an `ner` component, and trains it on the
auto-annotated examples (CSV labels + generic `TECH_TERM` context labels) using
spaCy's standard training loop (minibatching + dropout). Saves the best checkpoint
by dev F1.

Training on the `TECH_TERM` examples alongside the specific-category ones is what
teaches the model to flag terms it's never seen before, based on context — the
specific-category examples teach vocabulary, the `TECH_TERM` examples teach the
*pattern*.


In [9]:
def train_pipeline(train_examples, dev_examples, labels, cat_labels=None, n_iter=30):
    nlp = spacy.blank("en")
    ner = nlp.add_pipe("ner")
    for label in labels:
        ner.add_label(label)

    if cat_labels:
        textcat = nlp.add_pipe("textcat")
        for cat in cat_labels:
            textcat.add_label(cat)

    optimizer = nlp.begin_training()
    best_f1 = -1.0
    best_bytes = None

    for epoch in range(1, n_iter + 1):
        random.shuffle(train_examples)
        losses = {}
        batches = spacy.util.minibatch(train_examples, size=spacy.util.compounding(4.0, 32.0, 1.001))
        skipped = 0
        for batch in batches:
            try:
                nlp.update(batch, drop=0.2, losses=losses, sgd=optimizer)
            except ValueError:
                skipped += len(batch)
                continue

        dev_scored = [nlp(ex.reference.text) for ex in dev_examples]
        scored_examples = [
            Example.from_dict(
                pred,
                {
                    "entities": [(e.start_char, e.end_char, e.label_) for e in ref.reference.ents],
                    "cats": ref.reference.cats,
                }
            )
            for pred, ref in zip(dev_scored, dev_examples)
        ]
        ner_scores = Scorer.score_spans(scored_examples, "ents")
        ner_f1 = ner_scores.get("ents_f", 0.0) or 0.0

        msg = f"epoch {epoch:2d} | loss NER {losses.get('ner', 0.0):.1f}"
        if cat_labels:
            cat_scores = Scorer.score_cats(scored_examples, "cats", labels=cat_labels, multi_label=False)
            cat_f1 = cat_scores.get("cats_macro_f", 0.0) or 0.0
            msg += f", TextCat {losses.get('textcat', 0.0):.1f} | NER F1 {ner_f1:.3f} | TextCat F1 {cat_f1:.3f}"
            combined_f1 = (ner_f1 + cat_f1) / 2.0
        else:
            msg += f" | NER F1 {ner_f1:.3f}"
            combined_f1 = ner_f1

        if skipped:
            msg += f" (skipped {skipped} bad examples)"
        print(msg)

        if combined_f1 > best_f1:
            best_f1 = combined_f1
            best_bytes = nlp.to_bytes()

    if best_bytes is not None:
        nlp.from_bytes(best_bytes)
    print(f"Best combined dev score: {best_f1:.3f}")
    return nlp

labels = sorted(
    set(terms_df["label"].unique().tolist())
    | {GENERIC_LABEL}
    | {e.label_ for ex in all_examples for e in ex.reference.ents}
)
cat_labels = sorted({cat for ex in all_examples for cat in ex.reference.cats.keys()})
print(f"Labels to train: NER ({len(labels)}) = {labels}")
print(f"Task categories ({len(cat_labels)}) = {cat_labels}")

if len(train_examples) >= 10:
    ner_nlp = train_pipeline(train_examples, dev_examples, labels, cat_labels=cat_labels, n_iter=30)
else:
    print("Not enough documents yet to train a reliable model — "
          "add more OJT documents to data/ojt_documents/ and re-run. "
          "The rule-based ruler_nlp pipeline is still fully usable on its own.")
    ner_nlp = None


epoch  1 | loss 79875.20 | P 0.000 R 0.000 F1 0.000
epoch  2 | loss 1435.58 | P 0.000 R 0.000 F1 0.000
epoch  3 | loss 1497.97 | P 0.000 R 0.000 F1 0.000
epoch  4 | loss 1111.66 | P 0.750 R 0.010 F1 0.019
epoch  5 | loss 1667.58 | P 0.035 R 0.016 F1 0.022
epoch  6 | loss 969.27 | P 0.215 R 0.101 F1 0.137
epoch  7 | loss 867.84 | P 0.138 R 0.042 F1 0.065
epoch  8 | loss 863.23 | P 0.118 R 0.049 F1 0.069
epoch  9 | loss 788.22 | P 0.154 R 0.072 F1 0.098
epoch 10 | loss 691.11 | P 0.170 R 0.088 F1 0.116
epoch 11 | loss 636.95 | P 0.209 R 0.117 F1 0.150
epoch 12 | loss 554.87 | P 0.277 R 0.117 F1 0.165
epoch 13 | loss 508.83 | P 0.279 R 0.127 F1 0.174
epoch 14 | loss 440.42 | P 0.238 R 0.134 F1 0.171
epoch 15 | loss 468.91 | P 0.346 R 0.091 F1 0.144
epoch 16 | loss 388.02 | P 0.265 R 0.127 F1 0.172
epoch 17 | loss 348.83 | P 0.285 R 0.134 F1 0.182
epoch 18 | loss 366.23 | P 0.398 R 0.121 F1 0.185
epoch 19 | loss 456.89 | P 0.313 R 0.134 F1 0.187
epoch 20 | loss 300.51 | P 0.287 R 0.147 F1 

## Step 8 — Evaluate

Per-label precision / recall / F1 on the held-out dev set. Watch the `TECH_TERM`
row specifically — that's your signal for how well the model generalizes to
terms it wasn't explicitly trained on by name.


In [10]:
if ner_nlp is not None and dev_examples:
    preds = [ner_nlp(ex.reference.text) for ex in dev_examples]
    scored = [
        Example.from_dict(
            pred,
            {
                "entities": [(e.start_char, e.end_char, e.label_) for e in ref.reference.ents],
                "cats": ref.reference.cats,
            }
        )
        for pred, ref in zip(preds, dev_examples)
    ]
    ner_scores = Scorer.score_spans(scored, "ents")
    print("NER Overall:", {k: round(v, 3) for k, v in ner_scores.items() if k.startswith("ents_") and not isinstance(v, dict)})
    print("\nNER Per label:")
    for label, s in (ner_scores.get("ents_per_type") or {}).items():
        print(f"  {label:15s} P {s['p']:.3f}  R {s['r']:.3f}  F1 {s['f']:.3f}")

    if cat_labels:
        cat_scores = Scorer.score_cats(scored, "cats", labels=cat_labels, multi_label=False)
        print("\nTextCat (IT_TASK vs CLERICAL) Overall:")
        print(f"  Macro F1: {cat_scores.get('cats_macro_f', 0.0):.3f} | Macro P: {cat_scores.get('cats_macro_p', 0.0):.3f} | Macro R: {cat_scores.get('cats_macro_r', 0.0):.3f}")
        print("TextCat Per category:")
        for cat, s in (cat_scores.get("cats_f_per_type") or {}).items():
            print(f"  {cat:15s} P {s['p']:.3f}  R {s['r']:.3f}  F1 {s['f']:.3f}")


Overall: {'ents_p': 0.352, 'ents_r': 0.163, 'ents_f': 0.223}

Per label:
  TECH_TERM       P 0.270  R 0.113  F1 0.159
  FRAMEWORK       P 0.857  R 0.800  F1 0.828
  DATABASE        P 0.000  R 0.000  F1 0.000
  PROG_LANG       P 0.500  R 0.333  F1 0.400
  TOOL            P 0.429  R 0.750  F1 0.545
  CONCEPT         P 1.000  R 0.667  F1 0.800


### Step 8b — Test unknown / unseen tasks (Generalization Test)

This tests whether the trained model can recognize patterns in **brand-new tasks**
that never appeared in your training set, comparing the statistical `textcat` predictions
with the syntactic `classify_task_heuristic` from `context_miner.py`.


In [ ]:
from context_miner import classify_task_heuristic

test_unseen_tasks = [
    "Replaced the faulty RAM stick and applied thermal paste to the CPU",
    "Configured firewall rules and checked router LAN routing",
    "Shredded outdated student clearance records and arranged office cabinets",
    "Laminated visitor passes and attached IDs to lanyards for the event",
    "Set up dual monitor arms and connected HDMI cables to the workstations",
    "Organized incoming mail envelopes and routed vouchers for signature",
]

print("Generalization Test on Brand-New, Unseen Tasks:\n" + "=" * 65)
for task_text in test_unseen_tasks:
    doc = ner_nlp(task_text)
    heuristic = classify_task_heuristic(task_text)
    top_cat = max(doc.cats, key=doc.cats.get) if doc.cats else "N/A"
    conf = doc.cats[top_cat] if doc.cats else 0.0
    ents_found = [(e.text, e.label_) for e in doc.ents]
    print(f"Task: \"{task_text}\"")
    print(f"  → spaCy Model : [{top_cat}] (Confidence: {conf:.1%}) | Entities: {ents_found}")
    print(f"  → Heuristic   : [{max(heuristic, key=heuristic.get)}] (Confidence: {max(heuristic.values()):.1%})")
    print("-" * 65)


## Step 9 — Save artifacts

Both the rule-based pipeline and the trained NER pipeline are saved. The deployment
script can use either or both (hybrid mode is recommended: exact CSV matches for
precision + NER for recall on new/unlisted terms), and separately re-runs the
context-pattern miner from Step 4 — that part needs no trained model at all, so it
keeps surfacing brand-new terms even on day one.


In [11]:
ruler_nlp.to_disk(MODELS_DIR / "it_term_ruler")
print(f"Saved rule-based pipeline to {MODELS_DIR / 'it_term_ruler'}")

if ner_nlp is not None:
    ner_nlp.to_disk(MODELS_DIR / "it_term_ner")
    print(f"Saved trained NER pipeline to {MODELS_DIR / 'it_term_ner'}")

terms_df.to_csv(MODELS_DIR / "it_terms.csv", index=False)
print("Copied terms CSV alongside the models for deployment reference")


Saved rule-based pipeline to ../models/it_term_ruler
Saved trained NER pipeline to ../models/it_term_ner
Copied terms CSV alongside the models for deployment reference


## Next steps

- **Grow the CSV**: add new terms as students' documents reveal them — the top candidates
  table from Step 4 is your shortlist. The ruler pipeline picks up new patterns instantly
  (no retraining needed for exact matches).
- **Retrain periodically**: once you've promoted some `TECH_TERM` candidates to real
  categories in the CSV (and optionally corrected annotations from Step 5b), retrain —
  those terms move from generic `TECH_TERM` to their specific category, and the model
  keeps a fresh set of `TECH_TERM` examples to keep generalizing from.
- **Tune the trigger-verb list**: `context_miner.py`'s `TRIGGER_LEMMAS` list is a starting
  point — extend it with verbs you notice your students actually use ("familiarized with",
  "utilized", "was assigned to", etc.).
- **Use the deployment script** (`deploy_term_scanner.py`) to actually scan new OJT
  submissions and store/count matches to CSV — it also re-runs the context miner to keep
  surfacing brand-new candidate terms from every new batch of submissions.
